# LeetCode #142 — Linked List Cycle II: Floyd tìm điểm bắt đầu chu trình

**Mức độ:** Medium · **Chủ đề:** Hash Table, Linked List, Two Pointers

Skeleton trên LeetCode:

```python
class Solution:
    def detectCycle(self, head: Optional[ListNode]) -> Optional[ListNode]:
```

Mục tiêu của notebook:

1. Cách dễ: lưu các nút đã gặp vào **set** (`O(n)` bộ nhớ).
2. **Floyd (rùa và thỏ):** con chậm đi 1, con nhanh đi 2 — gặp nhau ⇔ có chu trình.
3. Chứng minh công thức **`a = k·c − b`**: từ điểm gặp và từ `head`, đi cùng tốc độ sẽ gặp nhau **đúng tại** đầu chu trình.
4. Mở rộng: tính độ dài chu trình, và ứng dụng #287 Find the Duplicate Number.

# 1. Bài toán là gì?

Cho `head` của danh sách liên kết. Trả về **nút** nơi chu trình bắt đầu; nếu không có chu trình, trả `None`. **Không** được sửa danh sách.

LeetCode mô tả đầu vào bằng `head` (danh sách giá trị) và `pos` — chỉ số nút mà nút **cuối** trỏ tới (`pos = −1` nghĩa là không có chu trình). `pos` **không** được truyền vào hàm.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 10^4]`.
- `-10^5 <= Node.val <= 10^5`
- `pos` là `-1` hoặc một chỉ số hợp lệ.

**Follow up (theo đề):** giải với bộ nhớ `O(1)` được không?

---

## Ví dụ

```text
head = [3, 2, 0, -4], pos = 1    ->  nút chỉ số 1 (giá trị 2)

  3 -> 2 -> 0 -> -4
       ^          |
       +----------+

head = [1, 2], pos = 0           ->  nút chỉ số 0
head = [1],    pos = -1          ->  None
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)
2. Hai con trỏ (two pointers)
3. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Hai con trỏ (two pointers)

### Ý tưởng

Dùng **hai chỉ số** di chuyển trên mảng/chuỗi theo quy tắc, để thay vòng lặp lồng `O(n²)` bằng một lượt `O(n)`. Hai kiểu chính:

**1. Hai đầu tiến vào nhau** (thường trên mảng đã sắp xếp):

```text
[1, 2, 4, 7, 11, 15], tìm hai số có tổng 15
 L                 R     1 + 15 = 16 > 15 → R lùi (cần nhỏ hơn)
 L             R         1 + 11 = 12 < 15 → L tiến (cần lớn hơn)
    L          R         2 + 11 = 13 < 15 → L tiến
       L       R         4 + 11 = 15 ✓
```

Vì sao được phép bỏ? Khi `a[L] + a[R] > target`, mọi cặp `(L', R)` với `L' > L` còn lớn hơn → `R` không thể là đáp án, bỏ `R`.

**2. Cùng chiều — con trỏ đọc và con trỏ ghi** (sửa mảng tại chỗ):

```text
xóa mọi số 0, giữ thứ tự: [0, 1, 0, 3, 12]
read đi qua từng phần tử; gặp số khác 0 thì chép vào vị trí write, write += 1
→ [1, 3, 12, ...]
```

In [6]:
def tong_hai_so_da_sap_xep(a, target):
    L, R = 0, len(a) - 1
    while L < R:
        s = a[L] + a[R]
        print(f"   L={L} R={R}: {a[L]} + {a[R]} = {s}")
        if s == target:
            return L, R
        if s < target:
            L += 1
        else:
            R -= 1
    return None

print("→", tong_hai_so_da_sap_xep([1, 2, 4, 7, 11, 15], 15))

def bo_so_khong(a):
    write = 0
    for read in range(len(a)):
        if a[read] != 0:
            a[write] = a[read]
            write += 1
    return a[:write]

print("bỏ số 0:", bo_so_khong([0, 1, 0, 3, 12]))

def la_palindrome(s):
    i, j = 0, len(s) - 1
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True
print("'racecar' palindrome?", la_palindrome("racecar"), "| 'abca'?", la_palindrome("abca"))

   L=0 R=5: 1 + 15 = 16
   L=0 R=4: 1 + 11 = 12
   L=1 R=4: 2 + 11 = 13
   L=2 R=4: 4 + 11 = 15
→ (2, 4)
bỏ số 0: [1, 3, 12]
'racecar' palindrome? True | 'abca'? False


### Đảo ngược một đoạn tại chỗ

Hai con trỏ ở hai đầu, đổi chỗ rồi tiến vào trong — dùng rất nhiều (đảo mảng, xoay mảng, đảo từ).

In [7]:
def dao_doan(a, i, j):
    while i < j:
        a[i], a[j] = a[j], a[i]
        i += 1; j -= 1

arr_tp = [1, 2, 3, 4, 5, 6]
dao_doan(arr_tp, 1, 4)
print("đảo đoạn [1..4]:", arr_tp)

đảo đoạn [1..4]: [1, 5, 4, 3, 2, 6]


### Lỗi hay gặp

- Quên **sắp xếp** trước khi dùng hai đầu cho bài tổng.
- Điều kiện `L <= R` thay vì `L < R` → dùng một phần tử hai lần.
- Không chứng minh được vì sao bỏ một đầu là an toàn → dễ bỏ mất đáp án.
- Con trỏ ghi vượt con trỏ đọc (không thể xảy ra nếu `write` chỉ tăng khi ghi).

## 2.3 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [8]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [9]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 447.09 ms | trên set: 0.079 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [10]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [11]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.4 Áp dụng vào bài này

Tìm nút bắt đầu chu trình. Cách dễ: **set** các nút đã đi qua, nút đầu tiên gặp lại là đáp án. Cách `O(1)`: **nhanh/chậm** (Floyd) — sau khi gặp nhau, đặt một con trỏ về `head`, hai con trỏ cùng đi 1 bước sẽ gặp nhau đúng ở điểm bắt đầu chu trình.

# 3. Nền tảng

In [12]:
from typing import List, Optional
import random


class ListNode:
    def __init__(self, x):
        self.val = x
        self.next = None


def build_cycle(values, pos):
    """Trả (head, list các nút) — nút cuối trỏ về nodes[pos] nếu pos >= 0."""
    nodes = [ListNode(v) for v in values]
    for a, b in zip(nodes, nodes[1:]):
        a.next = b
    if nodes and pos >= 0:
        nodes[-1].next = nodes[pos]
    return (nodes[0] if nodes else None), nodes

head, nodes = build_cycle([3, 2, 0, -4], 1)
print("nút cuối trỏ về:", nodes[-1].next.val)

nút cuối trỏ về: 2


# 4. Baseline: set các nút đã thăm

Nút đầu tiên được thăm **lần thứ hai** chính là đầu chu trình.

In [13]:
def detect_set(head: Optional[ListNode]) -> Optional[ListNode]:
    seen = set()
    node = head
    while node:
        if node in seen:
            return node
        seen.add(node)
        node = node.next
    return None

head, nodes = build_cycle([3, 2, 0, -4], 1)
r = detect_set(head)
print("Đầu chu trình:", r.val, "| là nodes[1]?", r is nodes[1])

Đầu chu trình: 2 | là nodes[1]? True


`O(n)` thời gian, `O(n)` bộ nhớ. Follow-up muốn `O(1)`.

# 5. Ý tưởng chính: Floyd — hai pha

**Pha 1 — phát hiện:** `slow` đi 1 bước, `fast` đi 2 bước. Nếu có chu trình, `fast` "đuổi kịp" `slow` từ phía sau (mỗi bước khoảng cách giảm 1) → hai con gặp nhau **bên trong** chu trình. Nếu `fast` chạm `None` → không có chu trình.

**Pha 2 — tìm đầu chu trình:** đặt một con trỏ về `head`, giữ con kia ở **điểm gặp**; cả hai đi **1 bước** mỗi lần. Chỗ chúng gặp nhau chính là đầu chu trình.

```text
a = số bước từ head tới đầu chu trình
c = độ dài chu trình
b = số bước từ đầu chu trình tới điểm gặp (đi theo chiều chu trình)
```

In [14]:
def detect_floyd(head: Optional[ListNode]) -> Optional[ListNode]:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            break
    else:
        return None                     # fast chạm None: không có chu trình
    p = head
    while p is not slow:
        p = p.next
        slow = slow.next
    return p

head, nodes = build_cycle([3, 2, 0, -4], 1)
print(detect_floyd(head) is nodes[1], detect_floyd(build_cycle([1], -1)[0]))

True None


`while … else`: khối `else` chỉ chạy khi vòng `while` kết thúc **do điều kiện sai** (không gặp `break`) — đúng trường hợp `fast` chạm cuối danh sách.

# 6. Vì sao pha 2 đúng? — chứng minh `a = k·c − b`

Khi gặp nhau: `slow` đã đi `a + b` bước (tính cả các vòng thừa, nhưng ta sẽ thấy `slow` đi chưa hết một vòng). `fast` đã đi gấp đôi: `2(a + b)`. `fast` đi hơn `slow` một số **nguyên** vòng:

```text
2(a + b) − (a + b) = k·c     (k ≥ 1)
⇒ a + b = k·c
⇒ a = k·c − b = (k − 1)·c + (c − b)
```

Từ điểm gặp, đi thêm `c − b` bước là tới đầu chu trình; đi thêm `(k−1)` vòng nữa vẫn ở đó. Vậy sau **đúng `a` bước**:

- con trỏ đi từ `head` tới đầu chu trình;
- con trỏ đi từ điểm gặp cũng tới đầu chu trình.

⇒ Chúng gặp nhau lần đầu đúng tại đầu chu trình.

# 7. Mô phỏng

In [15]:
def trace_floyd(values, pos):
    head, nodes = build_cycle(values, pos)
    idx = {id(n): i for i, n in enumerate(nodes)}
    slow = fast = head
    step = 0
    print("Pha 1:")
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        step += 1
        print(f"  bước {step}: slow ở chỉ số {idx[id(slow)]}, fast ở chỉ số {idx[id(fast)] if fast else None}")
        if slow is fast:
            print("  -> gặp nhau")
            break
    else:
        print("  fast chạm cuối -> không có chu trình")
        return None
    p, step = head, 0
    print("Pha 2:")
    while p is not slow:
        p = p.next; slow = slow.next; step += 1
        print(f"  bước {step}: p ở {idx[id(p)]}, slow ở {idx[id(slow)]}")
    print(f"Đầu chu trình: chỉ số {idx[id(p)]} (giá trị {p.val}); a = {step}")
    return p

_ = trace_floyd([3, 2, 0, -4], 1)
print()
_ = trace_floyd(list(range(10)), 3)

Pha 1:
  bước 1: slow ở chỉ số 1, fast ở chỉ số 2
  bước 2: slow ở chỉ số 2, fast ở chỉ số 1
  bước 3: slow ở chỉ số 3, fast ở chỉ số 3
  -> gặp nhau
Pha 2:
  bước 1: p ở 1, slow ở 1
Đầu chu trình: chỉ số 1 (giá trị 2); a = 1

Pha 1:
  bước 1: slow ở chỉ số 1, fast ở chỉ số 2
  bước 2: slow ở chỉ số 2, fast ở chỉ số 4
  bước 3: slow ở chỉ số 3, fast ở chỉ số 6
  bước 4: slow ở chỉ số 4, fast ở chỉ số 8
  bước 5: slow ở chỉ số 5, fast ở chỉ số 3
  bước 6: slow ở chỉ số 6, fast ở chỉ số 5
  bước 7: slow ở chỉ số 7, fast ở chỉ số 7
  -> gặp nhau
Pha 2:
  bước 1: p ở 1, slow ở 8
  bước 2: p ở 2, slow ở 9
  bước 3: p ở 3, slow ở 3
Đầu chu trình: chỉ số 3 (giá trị 3); a = 3


# 8. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def detectCycle(self, head: Optional[ListNode]) -> Optional[ListNode]:
        slow = fast = head
        while fast and fast.next:
            slow = slow.next
            fast = fast.next.next
            if slow is fast:
                p = head
                while p is not slow:
                    p = p.next
                    slow = slow.next
                return p
        return None

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


def run(values, pos):
    head, nodes = build_cycle(values, pos)
    r = Solution().detectCycle(head)
    # không được sửa danh sách: kiểm tra lại cấu trúc
    for a, b in zip(nodes, nodes[1:]):
        assert a.next is b
    if nodes:
        assert nodes[-1].next is (nodes[pos] if pos >= 0 else None)
    return nodes.index(r) if r else -1


check("Ví dụ 1", run([3, 2, 0, -4], 1), 1)
check("Ví dụ 2", run([1, 2], 0), 0)
check("Ví dụ 3", run([1], -1), -1)
check("Rỗng", run([], -1), -1)
check("Tự trỏ", run([7], 0), 0)
check("Vòng cả danh sách", run(list(range(6)), 0), 0)

random.seed(142)
for _ in range(5000):
    n = random.randint(0, 30)
    pos = random.randint(-1, n - 1) if n else -1
    assert run(list(range(n)), pos) == pos
print("5000 danh sách ngẫu nhiên: trả đúng chỉ số đầu chu trình, không sửa danh sách")

Ví dụ 1          PASS  1
Ví dụ 2          PASS  0
Ví dụ 3          PASS  -1
Rỗng             PASS  -1
Tự trỏ           PASS  0
Vòng cả danh sách PASS  0


5000 danh sách ngẫu nhiên: trả đúng chỉ số đầu chu trình, không sửa danh sách

# 9. Vì sao đúng? Độ phức tạp?

- **Pha 1 dừng:** khi cả hai đã vào chu trình, khoảng cách (theo chiều chu trình) từ `fast` tới `slow` giảm đúng 1 mỗi bước → gặp nhau sau tối đa `c` bước kể từ khi `slow` vào chu trình. Nên `slow` đi chưa hết một vòng: `b < c`.
- **Pha 2:** đã chứng minh ở mục 6.

```text
Time:  O(n)  (pha 1 ≤ a + c bước, pha 2 = a bước)
Space: O(1)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: So sánh giá trị thay vì đối tượng

In [18]:
def wrong_compare_val(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow.val == fast.val:                   # SAI: giá trị có thể trùng
            return True
    return False

head, _ = build_cycle([1, 2, 1, 2, 1], -1)
print("Sai : có chu trình?", wrong_compare_val(head), "| Đúng:", Solution().detectCycle(head) is not None)

Sai : có chu trình? True | Đúng: False


Dùng `is` để so sánh **đối tượng nút**.

## Lỗi 2: Kiểm tra gặp nhau **trước** khi di chuyển

`slow = fast = head` ban đầu đã bằng nhau — nếu kiểm tra trước khi đi, vòng lặp dừng ngay.

In [19]:
def wrong_check_before_move(head):
    slow = fast = head
    while fast and fast.next:
        if slow is fast:                           # SAI: luôn đúng ở lần đầu
            break
        slow = slow.next
        fast = fast.next.next
    else:
        return None
    p = head
    while p is not slow:
        p = p.next; slow = slow.next
    return p

head, nodes = build_cycle([1, 2, 3], -1)
print("Sai : danh sách không chu trình trả về nút", wrong_check_before_move(head).val, "| Đúng:", Solution().detectCycle(head))

Sai : danh sách không chu trình trả về nút 1 | Đúng: None


## Lỗi 3: Pha 2 cho cả hai con trỏ về `head`

Phải giữ **một** con ở điểm gặp. Đưa cả hai về `head` thì chúng trùng nhau ngay → luôn trả `head`.

## Lỗi 4: Điều kiện vòng `while fast.next and fast.next.next` mà quên `fast`

Khi danh sách rỗng, `fast` là `None` → `AttributeError`. Viết `while fast and fast.next`.

# 11. Mẹo nhận diện pattern

**Phát hiện chu trình trong dãy `x → f(x)`** (danh sách liên kết, hàm lặp, mảng chỉ số) → Floyd, `O(1)` bộ nhớ.

Bài liên quan: **#141 Linked List Cycle** (chỉ pha 1), **#287 Find the Duplicate Number** (mảng như danh sách: `i → nums[i]`), **#202 Happy Number** (chu trình của hàm "tổng bình phương chữ số"), **#876 Middle of the Linked List** (nhanh/chậm).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `values = 0..9`, `pos = 3`: tính `a`, `c`, điểm gặp `b`, và kiểm tra `a + b` là bội của `c`.

## Bài 2 — Điền chỗ trống

In [20]:
def exercise_2_detect_cycle(head: Optional[ListNode]) -> Optional[ListNode]:
    slow = fast = head
    while fast and fast.next:
        # TODO: di chuyển; nếu gặp nhau thì pha 2 và trả về
        pass
    return None


# h, ns = build_cycle([3, 2, 0, -4], 1); assert exercise_2_detect_cycle(h) is ns[1]
# h, ns = build_cycle([1], -1); assert exercise_2_detect_cycle(h) is None

## Bài 3 — Biến thể: độ dài chu trình

Trả về `c` (0 nếu không có chu trình), `O(1)` bộ nhớ.

In [21]:
def exercise_3_cycle_length(head: Optional[ListNode]) -> int:
    # TODO
    pass


# assert exercise_3_cycle_length(build_cycle([3, 2, 0, -4], 1)[0]) == 3
# assert exercise_3_cycle_length(build_cycle([1], -1)[0]) == 0
# assert exercise_3_cycle_length(build_cycle([7], 0)[0]) == 1

## Bài 4 — Biến thể: Find the Duplicate Number (LeetCode #287)

Mảng `nums` gồm `n + 1` số trong `[1, n]`, có **đúng một** giá trị lặp (có thể lặp nhiều lần). Tìm nó với `O(1)` bộ nhớ, **không** sửa mảng.

Gợi ý nhanh: coi mảng là danh sách liên kết `i → nums[i]`, bắt đầu từ chỉ số 0. Giá trị lặp chính là **đầu chu trình**.

```python
find_duplicate([1, 3, 4, 2, 2]) -> 2
find_duplicate([3, 1, 3, 4, 2]) -> 3
```

In [22]:
def exercise_4_find_duplicate(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_find_duplicate([1, 3, 4, 2, 2]) == 2
# assert exercise_4_find_duplicate([3, 1, 3, 4, 2]) == 3
# assert exercise_4_find_duplicate([2, 2, 2, 2, 2]) == 2

## Bài 5 — Mini challenge: Happy Number (LeetCode #202)

Thay `n` bằng tổng bình phương các chữ số; lặp lại. `n` "hạnh phúc" nếu đi tới 1. Kiểm tra bằng Floyd (`O(1)` bộ nhớ), không dùng set.

```python
is_happy(19) -> True     # 1² + 9² = 82 -> 68 -> 100 -> 1
is_happy(2)  -> False
```

In [23]:
def exercise_5_is_happy(n: int) -> bool:
    # TODO
    pass


# assert exercise_5_is_happy(19) is True
# assert exercise_5_is_happy(2) is False
# assert exercise_5_is_happy(1) is True

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_floyd(list(range(10)), 3)`; `a = 3`, `c = 7`.

**Bài 3**
- Mức 1: sau khi gặp nhau, giữ một con đứng yên, cho con kia đi tới khi quay lại — đếm số bước.

**Bài 4**
- Mức 1: `slow = fast = 0`; `slow = nums[slow]`, `fast = nums[nums[fast]]`.
- Mức 2: vì chỉ số 0 không phải giá trị nào trỏ tới (giá trị ∈ `[1, n]`), 0 nằm ngoài chu trình — giống `head`.

**Bài 5**
- Mức 1: `f(n) = sum(int(d)**2 for d in str(n))`; Floyd trên dãy `n, f(n), f(f(n)), …`; dãy luôn rơi vào chu trình (có thể là chu trình `1 → 1`).
- Mức 2: dừng khi `slow == fast`; trả `slow == 1`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
head, nodes = build_cycle(list(range(10)), 3)
slow = fast = head
steps = 0
while True:
    slow = slow.next; fast = fast.next.next; steps += 1
    if slow is fast:
        break
a, c = 3, 7
b = steps - a
print(f"slow đã đi a + b = {steps} bước -> b = {b}; (a + b) % c = {steps % c}")

slow đã đi a + b = 7 bước -> b = 4; (a + b) % c = 0


## Đáp án Bài 2

In [25]:
def answer_2_detect_cycle(head: Optional[ListNode]) -> Optional[ListNode]:
    return detect_floyd(head)

h, ns = build_cycle([3, 2, 0, -4], 1); assert answer_2_detect_cycle(h) is ns[1]
h, ns = build_cycle([1], -1); assert answer_2_detect_cycle(h) is None
print("Bài 2 PASS (giống detect_floyd ở mục 5)")

Bài 2 PASS (giống detect_floyd ở mục 5)


## Đáp án Bài 3

In [26]:
def answer_3_cycle_length(head: Optional[ListNode]) -> int:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            length, p = 1, slow.next
            while p is not slow:
                p = p.next
                length += 1
            return length
    return 0

assert answer_3_cycle_length(build_cycle([3, 2, 0, -4], 1)[0]) == 3
assert answer_3_cycle_length(build_cycle([1], -1)[0]) == 0
assert answer_3_cycle_length(build_cycle([7], 0)[0]) == 1
for _ in range(2000):
    n = random.randint(1, 20); pos = random.randint(-1, n - 1)
    assert answer_3_cycle_length(build_cycle(list(range(n)), pos)[0]) == (n - pos if pos >= 0 else 0)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_find_duplicate(nums: List[int]) -> int:
    slow = fast = 0
    while True:
        slow = nums[slow]
        fast = nums[nums[fast]]
        if slow == fast:
            break
    p = 0
    while p != slow:
        p = nums[p]
        slow = nums[slow]
    return p

assert answer_4_find_duplicate([1, 3, 4, 2, 2]) == 2
assert answer_4_find_duplicate([3, 1, 3, 4, 2]) == 3
assert answer_4_find_duplicate([2, 2, 2, 2, 2]) == 2
for _ in range(3000):
    n = random.randint(1, 15)
    dup = random.randint(1, n)
    nums = list(range(1, n + 1)) + [dup]
    for _ in range(random.randint(0, n - 1)):          # thay vài số bằng dup (vẫn chỉ một giá trị lặp)
        i = random.randrange(len(nums))
        nums[i] = dup
    random.shuffle(nums)
    assert answer_4_find_duplicate(nums) == dup
print("Bài 4 PASS")

Bài 4 PASS


Ở đây so sánh bằng `==` là đúng vì "nút" chính là **chỉ số** (số nguyên), không phải đối tượng.

## Đáp án Bài 5

In [28]:
def answer_5_is_happy(n: int) -> bool:
    def f(x):
        return sum(int(d) ** 2 for d in str(x))
    slow, fast = n, f(n)
    while slow != fast:
        slow = f(slow)
        fast = f(f(fast))
    return slow == 1

assert answer_5_is_happy(19) is True
assert answer_5_is_happy(2) is False
assert answer_5_is_happy(1) is True
print("Số hạnh phúc ≤ 50:", [k for k in range(1, 51) if answer_5_is_happy(k)])

Số hạnh phúc ≤ 50: [1, 7, 10, 13, 19, 23, 28, 31, 32, 44, 49]


# 15. Tổng kết

> Floyd: `slow` đi 1, `fast` đi 2; gặp nhau ⇔ có chu trình. Sau đó một con về `head`, con kia ở điểm gặp, cùng đi 1 bước — gặp nhau tại đầu chu trình (vì `a = k·c − b`). `O(n)` thời gian, `O(1)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] So sánh nút bằng `is` (không bằng `val`)?
- [ ] Di chuyển **trước**, kiểm tra gặp nhau **sau**?
- [ ] Pha 2: một con về `head`, một con giữ ở điểm gặp?
- [ ] `while fast and fast.next` (xử lý danh sách rỗng)?